In [1]:
from google.colab import drive
import os
import glob

drive.mount('/content/drive', force_remount=True)

# Reset về thư mục root của Colab
%cd /content/
if not os.path.exists('Clinical-Ambient-Documentation-Assistant'):
    !git clone https://github.com/DuyVuux/Clinical-Ambient-Documentation-Assistant.git

%cd /content/Clinical-Ambient-Documentation-Assistant

# Tìm file zip
possible_path = "/content/drive/MyDrive/Clinical Ambient Docs Assistant Data/data_lake.zip"
if not os.path.exists(possible_path):
    search = glob.glob("/content/drive/MyDrive/**/data_lake.zip", recursive=True)
    if search:
        possible_path = search[0]

if os.path.exists(possible_path):
    print(f"Đang giải nén dữ liệu từ: {possible_path}...")
    # Sử dụng tùy chọn -o để ghi đè và đảm bảo giải nén toàn bộ cấu trúc thư mục
    !unzip -o -q "{possible_path}" -d .
    print("Giải nén dữ liệu thành công!")
    # Kiểm tra lại xem thư mục audio đã xuất hiện chưa
    if os.path.exists('data/data_lake/silver/audio_clean'):
        print("Đã tìm thấy thư mục audio_clean.")
    else:
        print("CẢNH BÁO: Vẫn không thấy thư mục audio_clean sau khi giải nén!")
else:
    print("LỖI: Không tìm thấy file data_lake.zip!")

!pip install -r requirements.txt

Mounted at /content/drive
/content
Cloning into 'Clinical-Ambient-Documentation-Assistant'...
remote: Enumerating objects: 583, done.
remote: Counting objects: 100% (583/583), done.
remote: Compressing objects: 100% (349/349), done.
remote: Total 583 (delta 256), reused 513 (delta 186), pack-reused 0 (from 0)
Receiving objects: 100% (583/583), 2.11 MiB | 9.78 MiB/s, done.
Resolving deltas: 100% (256/256), done.
/content/Clinical-Ambient-Documentation-Assistant
Đang giải nén dữ liệu từ: /content/drive/MyDrive/Clinical Ambient Docs Assistant Data/data_lake.zip...
Giải nén dữ liệu thành công!
Đã tìm thấy thư mục audio_clean.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 91.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 99.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━

In [2]:
import json
import os

# Đảm bảo đang ở đúng thư mục dự án
%cd /content/Clinical-Ambient-Documentation-Assistant

original_manifest = 'data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl'
fixed_manifest = 'data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1_colab.jsonl'

# Định nghĩa các prefix cũ và mới
old_prefix = '/home/duykhongngu28/massive/Clinical Ambient Documentation Assistant/'
new_prefix = '/content/Clinical-Ambient-Documentation-Assistant/'

if os.path.exists(original_manifest):
    os.makedirs(os.path.dirname(fixed_manifest), exist_ok=True)
    with open(original_manifest, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    with open(fixed_manifest, 'w', encoding='utf-8') as f:
        for line in lines:
            data = json.loads(line)
            # Cập nhật tất cả các trường có thể chứa đường dẫn
            for key in ['audio_filepath', 'raw_audio_path', 'raw_transcript_path']:
                if key in data and isinstance(data[key], str):
                    data[key] = data[key].replace(old_prefix, new_prefix)
            f.write(json.dumps(data, ensure_ascii=False) + '\n')
    print(f'Đã cập nhật manifest tại: {os.path.abspath(fixed_manifest)}')
else:
    print(f'LỖI: Không tìm thấy manifest gốc tại {original_manifest}.')

/content/Clinical-Ambient-Documentation-Assistant
Đã cập nhật manifest tại: /content/Clinical-Ambient-Documentation-Assistant/data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1_colab.jsonl


## Update Code from Git

To ensure you have the latest code without re-running the model predictions, execute the cell below. It will navigate to your project directory and pull any new changes from the `main` branch.

In [28]:
import os

repo_path = '/content/Clinical-Ambient-Documentation-Assistant'

if not os.path.exists(repo_path):
    print(f"Repository not found. Cloning...")
    %cd /content/
    !git clone https://github.com/DuyVuux/Clinical-Ambient-Documentation-Assistant.git
    %cd {repo_path}
else:
    %cd {repo_path}
    print("Cập nhật bản mới nhất từ Git (Ghi đè bản cục bộ)... ")
    !git fetch origin main
    !git reset --hard origin/main

print("Đã đồng bộ thành công với Git.")

/content/Clinical-Ambient-Documentation-Assistant
Cập nhật bản mới nhất từ Git (Ghi đè bản cục bộ)... 
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 10 (delta 8), reused 10 (delta 8), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 1.46 KiB | 499.00 KiB/s, done.
From https://github.com/DuyVuux/Clinical-Ambient-Documentation-Assistant
 * branch            main       -> FETCH_HEAD
   16a5a6c..51fe87e  main       -> origin/main
HEAD is now at 51fe87e feat: Add padding-aware edge loss table to QA summary report
Đã đồng bộ thành công với Git.


## Chạy audit cho dev set

In [4]:
!python scripts/asr_preprocess/audit_audio_quality.py \
  --manifest data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1_colab.jsonl \
  --output_dir experiments/asr/preprocessing/audio_quality_audit/dev \
  --project_root "/content/Clinical-Ambient-Documentation-Assistant"

[OK] 1/200 public_vietmed_0884 warnings=[]
[OK] 2/200 public_vietmed_0174 warnings=[]
[OK] 3/200 public_vietmed_0148 warnings=[]
[OK] 4/200 public_vietmed_0420 warnings=[]
[OK] 5/200 public_vietmed_0962 warnings=[]
[OK] 6/200 public_vietmed_0149 warnings=[]
[OK] 7/200 public_vietmed_0976 warnings=['edge_loss_risk']
[OK] 8/200 public_vietmed_0080 warnings=[]
[OK] 9/200 public_vietmed_0776 warnings=[]
[OK] 10/200 public_vietmed_0303 warnings=[]
[OK] 11/200 public_vietmed_0996 warnings=[]
[OK] 12/200 public_vietmed_0526 warnings=['edge_loss_risk']
[OK] 13/200 public_vietmed_0775 warnings=[]
[OK] 14/200 public_vietmed_0232 warnings=[]
[OK] 15/200 public_vietmed_0626 warnings=[]
[OK] 16/200 public_vietmed_0801 warnings=[]
[OK] 17/200 public_vietmed_0562 warnings=[]
[OK] 18/200 public_vietmed_0510 warnings=['edge_loss_risk']
[OK] 19/200 public_vietmed_0841 warnings=[]
[OK] 20/200 public_vietmed_0495 warnings=[]
[OK] 21/200 public_vietmed_0120 warnings=[]
[OK] 22/200 public_vietmed_0478 warni

## Chạy audit cho train set

In [5]:
!python scripts/asr_preprocess/audit_audio_quality.py \
  --manifest data/data_lake/silver/asr_manifests/splits/vietmed_train_candidate_v0_1.jsonl \
  --output_dir experiments/asr/preprocessing/audio_quality_audit/train \
  --project_root "/content/Clinical-Ambient-Documentation-Assistant"

[OK] 1/600 public_vietmed_0852 warnings=[]
[OK] 2/600 public_vietmed_0346 warnings=[]
[OK] 3/600 public_vietmed_0404 warnings=[]
[OK] 4/600 public_vietmed_0227 warnings=[]
[OK] 5/600 public_vietmed_0988 warnings=[]
[OK] 6/600 public_vietmed_0165 warnings=[]
[OK] 7/600 public_vietmed_0336 warnings=[]
[OK] 8/600 public_vietmed_0601 warnings=['edge_loss_risk']
[OK] 9/600 public_vietmed_0917 warnings=[]
[OK] 10/600 public_vietmed_0473 warnings=[]
[OK] 11/600 public_vietmed_0448 warnings=[]
[OK] 12/600 public_vietmed_0108 warnings=[]
[OK] 13/600 public_vietmed_0857 warnings=[]
[OK] 14/600 public_vietmed_0156 warnings=[]
[OK] 15/600 public_vietmed_0459 warnings=[]
[OK] 16/600 public_vietmed_0281 warnings=[]
[OK] 17/600 public_vietmed_0208 warnings=[]
[OK] 18/600 public_vietmed_0155 warnings=[]
[OK] 19/600 public_vietmed_0588 warnings=[]
[OK] 20/600 public_vietmed_0758 warnings=[]
[OK] 21/600 public_vietmed_0986 warnings=[]
[OK] 22/600 public_vietmed_0384 warnings=[]
[OK] 23/600 public_vietme

# Preprocessing variants trên VietMed dev set


### Chạy debug 10 mẫu trước

In [7]:
!python scripts/asr_preprocess/build_preprocess_variants.py \
  --manifest data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl \
  --project_root "/content/Clinical-Ambient-Documentation-Assistant" \
  --variants p00_format_only p10_edge_pad_200ms p11_edge_pad_300ms p12_edge_pad_500ms \
  --max_samples 10

Building p00_format_only: 100% 10/10 [00:00<00:00, 142.45it/s]
{
  "variant": "p00_format_only",
  "input_manifest": "data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl",
  "output_manifest": "data/data_lake/silver/asr_manifests/preprocessing_variants/vietmed_dev_p00_format_only.jsonl",
  "n_input_rows": 10,
  "n_output_rows": 10,
  "n_failures": 0,
  "failure_path": "experiments/asr/preprocessing/day2_build_variants/p00_format_only_failures.jsonl"
}
Building p10_edge_pad_200ms: 100% 10/10 [00:00<00:00, 145.99it/s]
{
  "variant": "p10_edge_pad_200ms",
  "input_manifest": "data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl",
  "output_manifest": "data/data_lake/silver/asr_manifests/preprocessing_variants/vietmed_dev_p10_edge_pad_200ms.jsonl",
  "n_input_rows": 10,
  "n_output_rows": 10,
  "n_failures": 0,
  "failure_path": "experiments/asr/preprocessing/day2_build_variants/p10_edge_pad_200ms_failures.jsonl"
}
Building p11_edge_pad_300ms: 100% 10/10 [00:00<0

### Chạy padding-only full dev

In [8]:
!python scripts/asr_preprocess/build_preprocess_variants.py \
  --manifest data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl \
  --project_root "/content/Clinical-Ambient-Documentation-Assistant" \
  --variants p00_format_only p10_edge_pad_200ms p11_edge_pad_300ms p12_edge_pad_500ms

Building p00_format_only: 100% 200/200 [00:01<00:00, 134.92it/s]
{
  "variant": "p00_format_only",
  "input_manifest": "data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl",
  "output_manifest": "data/data_lake/silver/asr_manifests/preprocessing_variants/vietmed_dev_p00_format_only.jsonl",
  "n_input_rows": 200,
  "n_output_rows": 200,
  "n_failures": 0,
  "failure_path": "experiments/asr/preprocessing/day2_build_variants/p00_format_only_failures.jsonl"
}
Building p10_edge_pad_200ms: 100% 200/200 [00:01<00:00, 148.02it/s]
{
  "variant": "p10_edge_pad_200ms",
  "input_manifest": "data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl",
  "output_manifest": "data/data_lake/silver/asr_manifests/preprocessing_variants/vietmed_dev_p10_edge_pad_200ms.jsonl",
  "n_input_rows": 200,
  "n_output_rows": 200,
  "n_failures": 0,
  "failure_path": "experiments/asr/preprocessing/day2_build_variants/p10_edge_pad_200ms_failures.jsonl"
}
Building p11_edge_pad_300ms: 100% 200/20

### Chạy VAD variants nếu cài được Silero

In [9]:
!python scripts/asr_preprocess/build_preprocess_variants.py \
  --manifest data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl \
  --project_root "/content/Clinical-Ambient-Documentation-Assistant" \
  --variants p03_vad_pad_200ms p04_vad_pad_300ms p05_vad_pad_500ms \
  --enable_vad \
  --max_samples 10

[INFO] Loading Silero VAD...
Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip
[INFO] Silero VAD available: True
Building p03_vad_pad_200ms: 100% 10/10 [00:01<00:00,  7.09it/s]
{
  "variant": "p03_vad_pad_200ms",
  "input_manifest": "data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl",
  "output_manifest": "data/data_lake/silver/asr_manifests/preprocessing_variants/vietmed_dev_p03_vad_pad_200ms.jsonl",
  "n_input_rows": 10,
  "n_output_rows": 10,
  "n_failures": 0,
  "failure_path": "experiments/asr/preprocessing/day2_build_variants/p03_vad_pad_200ms_failures.jsonl"
}
Building p04_vad_pad_300ms: 100% 10/10 [00:01<00:00,  9.34it/s]
{
  "variant": "p04_vad_pad_300ms",
  "input_manifest": "data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl",
  "output_manifest": "data/data_lake/silver/asr_manifests/preprocessing_variants/vietmed_dev_p04_vad_pad_300ms.jsonl",
  "n_input_rows": 10,
  "n_output_rows": 10,
 

In [10]:
!python scripts/asr_preprocess/build_preprocess_variants.py \
  --manifest data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl \
  --project_root "/content/Clinical-Ambient-Documentation-Assistant" \
  --variants p03_vad_pad_200ms p04_vad_pad_300ms p05_vad_pad_500ms \
  --enable_vad

[INFO] Loading Silero VAD...
Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master
[INFO] Silero VAD available: True
Building p03_vad_pad_200ms: 100% 200/200 [00:25<00:00,  7.86it/s]
{
  "variant": "p03_vad_pad_200ms",
  "input_manifest": "data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl",
  "output_manifest": "data/data_lake/silver/asr_manifests/preprocessing_variants/vietmed_dev_p03_vad_pad_200ms.jsonl",
  "n_input_rows": 200,
  "n_output_rows": 200,
  "n_failures": 0,
  "failure_path": "experiments/asr/preprocessing/day2_build_variants/p03_vad_pad_200ms_failures.jsonl"
}
Building p04_vad_pad_300ms: 100% 200/200 [00:25<00:00,  7.89it/s]
{
  "variant": "p04_vad_pad_300ms",
  "input_manifest": "data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl",
  "output_manifest": "data/data_lake/silver/asr_manifests/preprocessing_variants/vietmed_dev_p04_vad_pad_300ms.jsonl",
  "n_input_rows": 200,
  "n_output_rows": 200,
  "n_failures": 0,
  "failur

In [13]:
import os
from google.colab import files

# 1. Thống kê lại các file hiện có
print("--- Danh sách toàn bộ file trong thư mục day2_build_variants ---")
!ls -lh experiments/asr/preprocessing/day2_build_variants/

# 2. Nén toàn bộ thư mục report/summary/failures
exp_dir = 'experiments/asr/preprocessing/day2_build_variants/'
zip_all_reports = 'day2_full_reports.zip'
!zip -j {zip_all_reports} {exp_dir}*

# 3. Nén các manifest JSONL kết quả (nếu chưa tải)
variant_dir = 'data/data_lake/silver/asr_manifests/preprocessing_variants/'
zip_variants = 'vietmed_dev_variants.zip'
!zip -j {zip_variants} {variant_dir}*.jsonl

# Tải các file về máy
for z in [zip_all_reports, zip_variants]:
    if os.path.exists(z):
        print(f"Đang chuẩn bị tải: {z}")
        files.download(z)
    else:
        print(f"Không tìm thấy {z} để tải.")

--- Danh sách toàn bộ file trong thư mục day2_build_variants ---
total 32K
-rw-r--r-- 1 root root 884 Jun  7 07:26 DAY2_BUILD_PREPROCESSING_VARIANTS_REPORT.md
-rw-r--r-- 1 root root   0 Jun  7 07:22 p00_format_only_failures.jsonl
-rw-r--r-- 1 root root 408 Jun  7 07:22 p00_format_only_summary.json
-rw-r--r-- 1 root root   0 Jun  7 07:26 p03_vad_pad_200ms_failures.jsonl
-rw-r--r-- 1 root root 414 Jun  7 07:26 p03_vad_pad_200ms_summary.json
-rw-r--r-- 1 root root   0 Jun  7 07:26 p04_vad_pad_300ms_failures.jsonl
-rw-r--r-- 1 root root 414 Jun  7 07:26 p04_vad_pad_300ms_summary.json
-rw-r--r-- 1 root root   0 Jun  7 07:26 p05_vad_pad_500ms_failures.jsonl
-rw-r--r-- 1 root root 414 Jun  7 07:26 p05_vad_pad_500ms_summary.json
-rw-r--r-- 1 root root   0 Jun  7 07:22 p10_edge_pad_200ms_failures.jsonl
-rw-r--r-- 1 root root 417 Jun  7 07:22 p10_edge_pad_200ms_summary.json
-rw-r--r-- 1 root root   0 Jun  7 07:22 p11_edge_pad_300ms_failures.jsonl
-rw-r--r-- 1 root root 417 Jun  7 07:22 p11_edge_

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Đang chuẩn bị tải: vietmed_dev_variants.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Day 3 week 4 - Audit preprocessing variants

### Chạy lại audit_audio_quality.py cho từng variant

In [19]:
!python scripts/asr_preprocess/audit_audio_quality.py \
  --manifest data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl \
  --output_dir experiments/asr/preprocessing/audio_quality_audit/variants/original \
  --project_root "$PROJECT_ROOT"

[OK] 1/200 public_vietmed_0884 warnings=[]
[OK] 2/200 public_vietmed_0174 warnings=[]
[OK] 3/200 public_vietmed_0148 warnings=[]
[OK] 4/200 public_vietmed_0420 warnings=[]
[OK] 5/200 public_vietmed_0962 warnings=[]
[OK] 6/200 public_vietmed_0149 warnings=[]
[OK] 7/200 public_vietmed_0976 warnings=['edge_loss_risk']
[OK] 8/200 public_vietmed_0080 warnings=[]
[OK] 9/200 public_vietmed_0776 warnings=[]
[OK] 10/200 public_vietmed_0303 warnings=[]
[OK] 11/200 public_vietmed_0996 warnings=[]
[OK] 12/200 public_vietmed_0526 warnings=['edge_loss_risk']
[OK] 13/200 public_vietmed_0775 warnings=[]
[OK] 14/200 public_vietmed_0232 warnings=[]
[OK] 15/200 public_vietmed_0626 warnings=[]
[OK] 16/200 public_vietmed_0801 warnings=[]
[OK] 17/200 public_vietmed_0562 warnings=[]
[OK] 18/200 public_vietmed_0510 warnings=['edge_loss_risk']
[OK] 19/200 public_vietmed_0841 warnings=[]
[OK] 20/200 public_vietmed_0495 warnings=[]
[OK] 21/200 public_vietmed_0120 warnings=[]
[OK] 22/200 public_vietmed_0478 warni

In [20]:
!python scripts/asr_preprocess/audit_audio_quality.py \
  --manifest data/data_lake/silver/asr_manifests/preprocessing_variants/vietmed_dev_p03_vad_pad_200ms.jsonl \
  --output_dir experiments/asr/preprocessing/audio_quality_audit/variants/p03_vad_pad_200ms \
  --project_root "$PROJECT_ROOT"

[OK] 1/200 public_vietmed_0884 warnings=[]
[OK] 2/200 public_vietmed_0174 warnings=[]
[OK] 3/200 public_vietmed_0148 warnings=[]
[OK] 4/200 public_vietmed_0420 warnings=[]
[OK] 5/200 public_vietmed_0962 warnings=[]
[OK] 6/200 public_vietmed_0149 warnings=[]
[OK] 7/200 public_vietmed_0976 warnings=['edge_loss_risk']
[OK] 8/200 public_vietmed_0080 warnings=['edge_loss_risk']
[OK] 9/200 public_vietmed_0776 warnings=[]
[OK] 10/200 public_vietmed_0303 warnings=[]
[OK] 11/200 public_vietmed_0996 warnings=[]
[OK] 12/200 public_vietmed_0526 warnings=['edge_loss_risk']
[OK] 13/200 public_vietmed_0775 warnings=['edge_loss_risk']
[OK] 14/200 public_vietmed_0232 warnings=[]
[OK] 15/200 public_vietmed_0626 warnings=[]
[OK] 16/200 public_vietmed_0801 warnings=[]
[OK] 17/200 public_vietmed_0562 warnings=[]
[OK] 18/200 public_vietmed_0510 warnings=['edge_loss_risk']
[OK] 19/200 public_vietmed_0841 warnings=[]
[OK] 20/200 public_vietmed_0495 warnings=[]
[OK] 21/200 public_vietmed_0120 warnings=[]
[OK] 

In [21]:
!python scripts/asr_preprocess/audit_audio_quality.py \
  --manifest data/data_lake/silver/asr_manifests/preprocessing_variants/vietmed_dev_p04_vad_pad_300ms.jsonl \
  --output_dir experiments/asr/preprocessing/audio_quality_audit/variants/p04_vad_pad_300ms \
  --project_root "$PROJECT_ROOT"

[OK] 1/200 public_vietmed_0884 warnings=[]
[OK] 2/200 public_vietmed_0174 warnings=[]
[OK] 3/200 public_vietmed_0148 warnings=[]
[OK] 4/200 public_vietmed_0420 warnings=[]
[OK] 5/200 public_vietmed_0962 warnings=[]
[OK] 6/200 public_vietmed_0149 warnings=[]
[OK] 7/200 public_vietmed_0976 warnings=['edge_loss_risk']
[OK] 8/200 public_vietmed_0080 warnings=[]
[OK] 9/200 public_vietmed_0776 warnings=[]
[OK] 10/200 public_vietmed_0303 warnings=[]
[OK] 11/200 public_vietmed_0996 warnings=[]
[OK] 12/200 public_vietmed_0526 warnings=['edge_loss_risk']
[OK] 13/200 public_vietmed_0775 warnings=['edge_loss_risk']
[OK] 14/200 public_vietmed_0232 warnings=[]
[OK] 15/200 public_vietmed_0626 warnings=[]
[OK] 16/200 public_vietmed_0801 warnings=[]
[OK] 17/200 public_vietmed_0562 warnings=[]
[OK] 18/200 public_vietmed_0510 warnings=['edge_loss_risk']
[OK] 19/200 public_vietmed_0841 warnings=[]
[OK] 20/200 public_vietmed_0495 warnings=[]
[OK] 21/200 public_vietmed_0120 warnings=[]
[OK] 22/200 public_vi

In [22]:
!python scripts/asr_preprocess/audit_audio_quality.py \
  --manifest data/data_lake/silver/asr_manifests/preprocessing_variants/vietmed_dev_p05_vad_pad_500ms.jsonl \
  --output_dir experiments/asr/preprocessing/audio_quality_audit/variants/p05_vad_pad_500ms \
  --project_root "$PROJECT_ROOT"

[OK] 1/200 public_vietmed_0884 warnings=[]
[OK] 2/200 public_vietmed_0174 warnings=[]
[OK] 3/200 public_vietmed_0148 warnings=[]
[OK] 4/200 public_vietmed_0420 warnings=[]
[OK] 5/200 public_vietmed_0962 warnings=[]
[OK] 6/200 public_vietmed_0149 warnings=[]
[OK] 7/200 public_vietmed_0976 warnings=['edge_loss_risk']
[OK] 8/200 public_vietmed_0080 warnings=[]
[OK] 9/200 public_vietmed_0776 warnings=[]
[OK] 10/200 public_vietmed_0303 warnings=[]
[OK] 11/200 public_vietmed_0996 warnings=[]
[OK] 12/200 public_vietmed_0526 warnings=['edge_loss_risk']
[OK] 13/200 public_vietmed_0775 warnings=[]
[OK] 14/200 public_vietmed_0232 warnings=[]
[OK] 15/200 public_vietmed_0626 warnings=[]
[OK] 16/200 public_vietmed_0801 warnings=[]
[OK] 17/200 public_vietmed_0562 warnings=[]
[OK] 18/200 public_vietmed_0510 warnings=['edge_loss_risk']
[OK] 19/200 public_vietmed_0841 warnings=[]
[OK] 20/200 public_vietmed_0495 warnings=[]
[OK] 21/200 public_vietmed_0120 warnings=[]
[OK] 22/200 public_vietmed_0478 warni

In [23]:
import os
from google.colab import files

# Đường dẫn thư mục chứa các kết quả audit của variants
audit_variants_dir = 'experiments/asr/preprocessing/audio_quality_audit/variants/'
zip_filename = 'audio_quality_audit_variants_results.zip'

if os.path.exists(audit_variants_dir):
    print(f"--- Đang nén các kết quả audit từ {audit_variants_dir} ---")
    # Nén cả thư mục variants (bao gồm subfolders original, p03, p04, p05)
    !zip -r {zip_filename} {audit_variants_dir}

    if os.path.exists(zip_filename):
        print(f"Đang chuẩn bị tải: {zip_filename}")
        files.download(zip_filename)
    else:
        print("Lỗi: Không tạo được file zip.")
else:
    print(f"LỖI: Không tìm thấy thư mục {audit_variants_dir}")

--- Đang nén các kết quả audit từ experiments/asr/preprocessing/audio_quality_audit/variants/ ---
  adding: experiments/asr/preprocessing/audio_quality_audit/variants/ (stored 0%)
  adding: experiments/asr/preprocessing/audio_quality_audit/variants/p04_vad_pad_300ms/ (stored 0%)
  adding: experiments/asr/preprocessing/audio_quality_audit/variants/p04_vad_pad_300ms/failed_audio_quality_audit.jsonl (stored 0%)
  adding: experiments/asr/preprocessing/audio_quality_audit/variants/p04_vad_pad_300ms/audio_quality_report.json (deflated 82%)
  adding: experiments/asr/preprocessing/audio_quality_audit/variants/p04_vad_pad_300ms/edge_loss_risk_samples.jsonl (deflated 77%)
  adding: experiments/asr/preprocessing/audio_quality_audit/variants/p04_vad_pad_300ms/DAY1_AUDIO_QA_REPORT.md (deflated 56%)
  adding: experiments/asr/preprocessing/audio_quality_audit/variants/p05_vad_pad_500ms/ (stored 0%)
  adding: experiments/asr/preprocessing/audio_quality_audit/variants/p05_vad_pad_500ms/failed_audio_qua

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Tạo padding-only variants

In [26]:
!python scripts/asr_preprocess/build_preprocess_variants.py \
  --manifest data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl \
  --project_root "/content/Clinical-Ambient-Documentation-Assistant" \
  --variants p10_edge_pad_200ms p11_edge_pad_300ms p12_edge_pad_500ms

Building p10_edge_pad_200ms: 100% 200/200 [00:01<00:00, 144.32it/s]
{
  "variant": "p10_edge_pad_200ms",
  "input_manifest": "data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl",
  "output_manifest": "data/data_lake/silver/asr_manifests/preprocessing_variants/vietmed_dev_p10_edge_pad_200ms.jsonl",
  "n_input_rows": 200,
  "n_output_rows": 200,
  "n_failures": 0,
  "failure_path": "experiments/asr/preprocessing/day2_build_variants/p10_edge_pad_200ms_failures.jsonl"
}
Building p11_edge_pad_300ms: 100% 200/200 [00:01<00:00, 144.79it/s]
{
  "variant": "p11_edge_pad_300ms",
  "input_manifest": "data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl",
  "output_manifest": "data/data_lake/silver/asr_manifests/preprocessing_variants/vietmed_dev_p11_edge_pad_300ms.jsonl",
  "n_input_rows": 200,
  "n_output_rows": 200,
  "n_failures": 0,
  "failure_path": "experiments/asr/preprocessing/day2_build_variants/p11_edge_pad_300ms_failures.jsonl"
}
Building p12_edge_pad_500ms:

## Audit p10/p11/p12 bằng script

In [27]:
PROJECT_ROOT="/content/Clinical-Ambient-Documentation-Assistant"

!python scripts/asr_preprocess/audit_audio_quality.py \
  --manifest data/data_lake/silver/asr_manifests/preprocessing_variants/vietmed_dev_p10_edge_pad_200ms.jsonl \
  --output_dir experiments/asr/preprocessing/audio_quality_audit/variants/p10_edge_pad_200ms \
  --project_root "$PROJECT_ROOT"

!python scripts/asr_preprocess/audit_audio_quality.py \
  --manifest data/data_lake/silver/asr_manifests/preprocessing_variants/vietmed_dev_p11_edge_pad_300ms.jsonl \
  --output_dir experiments/asr/preprocessing/audio_quality_audit/variants/p11_edge_pad_300ms \
  --project_root "$PROJECT_ROOT"

!python scripts/asr_preprocess/audit_audio_quality.py \
  --manifest data/data_lake/silver/asr_manifests/preprocessing_variants/vietmed_dev_p12_edge_pad_500ms.jsonl \
  --output_dir experiments/asr/preprocessing/audio_quality_audit/variants/p12_edge_pad_500ms \
  --project_root "$PROJECT_ROOT"

[OK] 1/200 public_vietmed_0884 warnings=['edge_loss_risk']
[OK] 2/200 public_vietmed_0174 warnings=['edge_loss_risk']
[OK] 3/200 public_vietmed_0148 warnings=[]
[OK] 4/200 public_vietmed_0420 warnings=[]
[OK] 5/200 public_vietmed_0962 warnings=['edge_loss_risk']
[OK] 6/200 public_vietmed_0149 warnings=['edge_loss_risk']
[OK] 7/200 public_vietmed_0976 warnings=['edge_loss_risk']
[OK] 8/200 public_vietmed_0080 warnings=['edge_loss_risk']
[OK] 9/200 public_vietmed_0776 warnings=[]
[OK] 10/200 public_vietmed_0303 warnings=[]
[OK] 11/200 public_vietmed_0996 warnings=[]
[OK] 12/200 public_vietmed_0526 warnings=['edge_loss_risk']
[OK] 13/200 public_vietmed_0775 warnings=[]
[OK] 14/200 public_vietmed_0232 warnings=[]
[OK] 15/200 public_vietmed_0626 warnings=['edge_loss_risk']
[OK] 16/200 public_vietmed_0801 warnings=[]
[OK] 17/200 public_vietmed_0562 warnings=[]
[OK] 18/200 public_vietmed_0510 warnings=['edge_loss_risk']
[OK] 19/200 public_vietmed_0841 warnings=['edge_loss_risk']
[OK] 20/200 p

## Phân tích - QA

In [29]:
!python scripts/asr_preprocess/summarize_variant_audio_qa.py \
    --project_root "/content/Clinical-Ambient-Documentation-Assistant"

[DONE] JSON: /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/preprocessing/audio_quality_audit/variant_audio_qa_summary.json
[DONE] MD:   /content/Clinical-Ambient-Documentation-Assistant/experiments/asr/preprocessing/audio_quality_audit/VARIANT_AUDIO_QA_SUMMARY.md


## Chạy ChunkFormer trên 3 variant

In [30]:
!python scripts/asr_eval/run_chunkformer_ctc.py \
  --manifest data/data_lake/silver/asr_manifests/splits/vietmed_dev_v0_1.jsonl \
  --model khanhld/chunkformer-ctc-large-vie \
  --output experiments/asr/preprocessing/ablation/predictions/dev/chunkformer_original_dev_predictions.jsonl

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):
[INFO] Loading ChunkFormer model: khanhld/chunkformer-ctc-large-vie
Fetching 10 files:   0% 0/10 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Fetching 10 files: 100% 10/10 [00:09<00:00,  1.11it/s]
Download complete: 100% 615M/615M [00:09<00:0

In [31]:
!python scripts/asr_eval/run_chunkformer_ctc.py \
  --manifest data/data_lake/silver/asr_manifests/preprocessing_variants/vietmed_dev_p10_edge_pad_200ms.jsonl \
  --model khanhld/chunkformer-ctc-large-vie \
  --output experiments/asr/preprocessing/ablation/predictions/dev/chunkformer_p10_edge_pad_200ms_dev_predictions.jsonl

[INFO] Loading ChunkFormer model: khanhld/chunkformer-ctc-large-vie
Fetching 10 files: 100% 10/10 [00:00<00:00, 79437.58it/s]
Download complete: : 0.00B [00:00, ?B/s]
  0% 0/1 [00:00<?, ?it/s]
[1/200] public_vietmed_0884
  0% 0/1 [00:00<?, ?it/s]
[2/200] public_vietmed_0174
  0% 0/1 [00:00<?, ?it/s]
[3/200] public_vietmed_0148
  0% 0/1 [00:00<?, ?it/s]
[4/200] public_vietmed_0420
  0% 0/1 [00:00<?, ?it/s]
[5/200] public_vietmed_0962
  0% 0/1 [00:00<?, ?it/s]
[6/200] public_vietmed_0149
  0% 0/1 [00:00<?, ?it/s]
[7/200] public_vietmed_0976
  0% 0/1 [00:00<?, ?it/s]
[8/200] public_vietmed_0080
  0% 0/1 [00:00<?, ?it/s]
[9/200] public_vietmed_0776
  0% 0/1 [00:00<?, ?it/s]
[10/200] public_vietmed_0303
  0% 0/1 [00:00<?, ?it/s]
[11/200] public_vietmed_0996
  0% 0/1 [00:00<?, ?it/s]
[12/200] public_vietmed_0526
  0% 0/1 [00:00<?, ?it/s]
[13/200] public_vietmed_0775
  0% 0/1 [00:00<?, ?it/s]
[14/200] public_vietmed_0232
  0% 0/1 [00:00<?, ?it/s]
[15/200] public_vietmed_0626
  0% 0/1 [00:00<?

In [32]:
!python scripts/asr_eval/run_chunkformer_ctc.py \
  --manifest data/data_lake/silver/asr_manifests/preprocessing_variants/vietmed_dev_p03_vad_pad_200ms.jsonl \
  --model khanhld/chunkformer-ctc-large-vie \
  --output experiments/asr/preprocessing/ablation/predictions/dev/chunkformer_p03_vad_pad_200ms_dev_predictions.jsonl

[INFO] Loading ChunkFormer model: khanhld/chunkformer-ctc-large-vie
Fetching 10 files: 100% 10/10 [00:00<00:00, 105120.40it/s]
Download complete: : 0.00B [00:00, ?B/s]
  0% 0/1 [00:00<?, ?it/s]
[1/200] public_vietmed_0884
  0% 0/1 [00:00<?, ?it/s]
[2/200] public_vietmed_0174
  0% 0/1 [00:00<?, ?it/s]
[3/200] public_vietmed_0148
  0% 0/1 [00:00<?, ?it/s]
[4/200] public_vietmed_0420
  0% 0/1 [00:00<?, ?it/s]
[5/200] public_vietmed_0962
  0% 0/1 [00:00<?, ?it/s]
[6/200] public_vietmed_0149
  0% 0/1 [00:00<?, ?it/s]
[7/200] public_vietmed_0976
  0% 0/1 [00:00<?, ?it/s]
[8/200] public_vietmed_0080
  0% 0/1 [00:00<?, ?it/s]
[9/200] public_vietmed_0776
  0% 0/1 [00:00<?, ?it/s]
[10/200] public_vietmed_0303
  0% 0/1 [00:00<?, ?it/s]
[11/200] public_vietmed_0996
  0% 0/1 [00:00<?, ?it/s]
[12/200] public_vietmed_0526
  0% 0/1 [00:00<?, ?it/s]
[13/200] public_vietmed_0775
  0% 0/1 [00:00<?, ?it/s]
[14/200] public_vietmed_0232
  0% 0/1 [00:00<?, ?it/s]
[15/200] public_vietmed_0626
  0% 0/1 [00:00<

In [33]:
!python scripts/asr_eval/compute_wer.py \
  --predictions experiments/asr/preprocessing/ablation/predictions/dev/chunkformer_original_dev_predictions.jsonl \
  --output experiments/asr/preprocessing/ablation/metrics/dev/chunkformer_original_dev_metrics.json

!python scripts/asr_eval/compute_wer.py \
  --predictions experiments/asr/preprocessing/ablation/predictions/dev/chunkformer_p10_edge_pad_200ms_dev_predictions.jsonl \
  --output experiments/asr/preprocessing/ablation/metrics/dev/chunkformer_p10_edge_pad_200ms_dev_metrics.json

!python scripts/asr_eval/compute_wer.py \
  --predictions experiments/asr/preprocessing/ablation/predictions/dev/chunkformer_p03_vad_pad_200ms_dev_predictions.jsonl \
  --output experiments/asr/preprocessing/ablation/metrics/dev/chunkformer_p03_vad_pad_200ms_dev_metrics.json

{
  "n_samples": 200,
  "strict_wer": 0.12900718865358463,
  "normalized_wer": 0.12900718865358463,
  "strict_cer": 0.12032833493832255,
  "normalized_cer": 0.12032833493832255
}
{
  "n_samples": 200,
  "strict_wer": 0.14493879930056344,
  "normalized_wer": 0.14493879930056344,
  "strict_cer": 0.13559866098041914,
  "normalized_cer": 0.13559866098041914
}
{
  "n_samples": 200,
  "strict_wer": 0.12900718865358463,
  "normalized_wer": 0.12900718865358463,
  "strict_cer": 0.12060347594809007,
  "normalized_cer": 0.12060347594809007
}
